# step01_suna_spectral_qc - SUNA Stage 1: per-frame spectral QC (P45_06)

Stage 0 asked "what is in each file and is the instrument healthy". Stage 1 asks the next question, frame by
frame: **is this spectrum trustworthy enough to turn into a nitrate value?** It takes the raw frames, builds
the per-frame absorbance, runs a set of physically motivated spectral checks, and writes a Level-1 table where
every frame is either clean or carries a reason it is not.

**Why spectral QC is the right second step.** A SUNA nitrate is the output of a least-squares fit of the
217 to 240 nm absorbance to reference spectra for nitrate, bromide and a linear baseline (Johnson and Coletti,
2002; Sakamoto, Johnson and Coletti, 2009). The fit will always return a number, even from a spectrum taken in
air, through a bubble, or with a fouled window. So a plausible-looking nitrate is not evidence the measurement
was good. Stage 1 checks the spectrum itself, which is the only honest way to know (Johnson et al., 2018).

**What Stage 1 does NOT do (on purpose).** It does not split the continuous logs into casts and it does not
assign depth. From Stage 0 we learned the P45_06 files are continuous across several casts, with the instrument
left in the water between them, so there are no clean gaps to cut on. Splitting and depth both need the CTD
time-and-pressure record, which comes in at co-location. Until then Stage 1 only *flags*; it deletes nothing, so
no frame is lost before we have the CTD timebase. The long between-cast baseline stretches are kept and simply
marked, not removed.

**The checks, and the reasoning behind each (thresholds derived from your own P45_06 data, not guessed):**
- **out of water** (optical gate): in seawater the lamp light is attenuated to a stable level and the baseline
  absorbance sits near a fixed value; in air or through a bubble it is not. Frames outside peak 20000 to 58000
  counts, `abs254` outside -0.6 to -0.15, or `spec_avg` outside 8000 to 30000 are flagged. This catches the
  recovery and surface-break frames.
- **saturation in the fit window**: any 217 to 240 nm channel at the 16-bit ceiling makes the fit meaningless.
- **low signal / high RMSE / warm-up / humidity**: carried from Stage 0 (lamp or optics problems, poor fits).
- **long-wave baseline anomaly**: nitrate and bromide do not absorb above ~310 nm, so clean water shows near-zero
  absorbance there. Mean `|absorbance|` over 310 to 360 nm above 0.10 means turbidity, CDOM or a bubble has
  tilted the baseline, which biases the fit.
- **shape anomaly**: a frame whose spectrum shape (250 to 360 nm, OUTSIDE the nitrate band so real nitrate does
  not trip it) departs from the clean reference by a correlation drop above 0.01 has a broadband distortion
  (bubble, fouling). Measuring this outside the fit window is deliberate: a within-window shape test would reject
  real high-nitrate frames, which is the opposite of what we want.

A frame is **L1 clean** when none of the fatal flags fire (humidity is advisory). The interactive explorer lets
you step to the next flagged frame and see exactly why it failed: raw spectrum, absorbance near the fit window,
and the long-wave baseline.

Sources: Johnson and Coletti (2002), *DSR I*; Sakamoto, Johnson and Coletti (2009), *L&O: Methods*;
Johnson et al. (2018), *JTECH* (BGC-Argo nitrate QC and spectral baseline checks); Sea-Bird SUNA V2 manual.

In [ ]:
import re, sys
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib

INTERACTIVE_ZOOM = True
try:
    get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM = False
    try: get_ipython().run_line_magic("matplotlib", "inline")
    except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try:
    from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image = None

# ------------------------------------------------------------------ resolve repo root
try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root, find_suna_cal

# ------------------------------------------------------------------ config
CRUISE_ID  = "P45_06"
CTD_ROOT   = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
SUNA_L0    = CRUISE_DIR / "L0" / "SUNA"
CAL_FILE   = find_suna_cal(cruise_id=CRUISE_ID, filename="SNA2503C.cal")
OUT_DIR    = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)
L1_DIR     = CRUISE_DIR / "L1" / "SUNA";     L1_DIR.mkdir(parents=True, exist_ok=True)

NSPEC = 256; SPEC_START = 11
FIT_LO, FIT_HI = 217.0, 240.0

SAT_COUNTS = 60000; LOW_COUNTS = 2000; RMSE_HI = 1.0e-3; HUM_HI = 10.0; WARMUP_S = 60.0
PEAK_LO, PEAK_HI       = 20000, 58000
ABS254_LO, ABS254_HI   = -0.60, -0.15
SPECAVG_LO, SPECAVG_HI = 8000, 30000
LONGWAVE_LO, LONGWAVE_HI = 310.0, 360.0
LONGWAVE_A_HI          = 0.10
SHAPE_LO, SHAPE_HI     = 250.0, 360.0
SHAPE_DEV_HI           = 0.01

EXPLORER_FIGSIZE = (13.5, 5.2)
print("cruise:", CRUISE_ID, "| L0:", SUNA_L0, "| L1 out:", L1_DIR)

In [ ]:
# ------------------------------------------------------------------ readers (standalone, same as Stage 0)
def parse_cal(path):
    wl = []
    try: lines = Path(path).read_text(errors="replace").splitlines()
    except Exception as e:
        print("cal read failed:", e); return np.arange(NSPEC, dtype=float)
    for ln in lines:
        p = ln.split(",")
        if p[0] == "E" and len(p) >= 2:
            try: wl.append(float(p[1]))
            except ValueError: pass
    return np.array(wl, float)

def _ts(datef, timed):
    y = int(float(datef)) // 1000; doy = int(float(datef)) % 1000
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=float(timed))

def parse_suna_file(path):
    fhr, rows, spec = {}, [], []
    for ln in Path(path).read_text(errors="replace").splitlines():
        if ln.startswith("SATFHR"):
            p = ln.split(",", 2)
            if len(p) == 3: fhr[p[1].strip()] = p[2].strip()
        elif ln.startswith("SATSLF"):
            f = ln.split(",")
            if len(f) < SPEC_START + NSPEC + 6: continue
            try: sp = np.array(f[SPEC_START:SPEC_START + NSPEC], float)
            except ValueError: continue
            tr = f[SPEC_START + NSPEC:]
            def gg(i):
                try: return float(tr[i])
                except (ValueError, IndexError): return np.nan
            try:
                rows.append(dict(time_utc=_ts(f[1], f[2]),
                    nitrate_uM=float(f[3]), abs254=float(f[5]), abs350=float(f[6]),
                    bromide=float(f[7]), spec_avg=float(f[8]), dark=float(f[9]),
                    T_int=gg(0), T_lamp=gg(2), lamp_s=gg(3), humidity=gg(4),
                    I_main=gg(8), fit_rmse=gg(13)))
                spec.append(sp)
            except ValueError: continue
    return fhr, pd.DataFrame(rows), (np.array(spec) if spec else np.empty((0, NSPEC)))

def qc_frames(df, spec, wl):
    """Add L0 integrity + L1 spectral QC flags. Returns df with flags and L1_clean."""
    if df.empty: return df
    df = df.copy()
    t0 = df["time_utc"].iloc[0]
    secs = (df["time_utc"] - t0).dt.total_seconds().to_numpy()
    peak = spec.max(axis=1)
    df["peak_counts"] = peak
    # L0 integrity
    df["f_warmup"]    = secs < WARMUP_S
    df["f_saturated"] = peak > SAT_COUNTS
    df["f_lowsignal"] = peak < LOW_COUNTS
    df["f_highrmse"]  = df["fit_rmse"] > RMSE_HI
    df["f_humid"]     = df["humidity"] > HUM_HI
    _l0bad = df[["f_warmup","f_saturated","f_lowsignal","f_highrmse","f_humid"]].any(axis=1).to_numpy()
    # clean reference = median over L0-clean frames (fallback: all)
    ref = np.median(spec[~_l0bad], axis=0) if (~_l0bad).sum() >= 5 else np.median(spec, axis=0)
    dk = float(np.median(df["dark"]))
    num = np.clip(spec - dk, 1, None); den = np.clip(ref - dk, 1, None)
    A = -np.log10(num / den)
    # L1 spectral checks
    lw = (wl >= LONGWAVE_LO) & (wl <= LONGWAVE_HI)
    sb = (wl >= SHAPE_LO) & (wl <= SHAPE_HI)
    fitb = (wl >= FIT_LO) & (wl <= FIT_HI)
    df["longwave_A"] = np.mean(np.abs(A[:, lw]), axis=1)
    rb = ref[sb] - ref[sb].mean()
    sbn = spec[:, sb] - spec[:, sb].mean(axis=1, keepdims=True)
    corr = (sbn @ rb) / (np.sqrt((sbn**2).sum(1)) * np.sqrt((rb**2).sum()) + 1e-9)
    df["shape_dev"] = 1.0 - corr
    df["f_outofwater"] = ~(df["peak_counts"].between(PEAK_LO, PEAK_HI)
                           & df["abs254"].between(ABS254_LO, ABS254_HI)
                           & df["spec_avg"].between(SPECAVG_LO, SPECAVG_HI))
    df["f_satfit"]   = spec[:, fitb].max(axis=1) >= SAT_COUNTS
    df["f_baseline"] = df["longwave_A"] > LONGWAVE_A_HI
    df["f_shape"]    = df["shape_dev"] > SHAPE_DEV_HI
    FATAL = ["f_warmup","f_saturated","f_lowsignal","f_highrmse",
             "f_outofwater","f_satfit","f_baseline","f_shape"]
    df["L1_clean"] = ~df[FATAL].any(axis=1)
    return df

FATAL_FLAGS = ["f_outofwater","f_satfit","f_saturated","f_lowsignal",
               "f_highrmse","f_baseline","f_shape","f_warmup"]
ALL_FLAGS = FATAL_FLAGS + ["f_humid"]

In [ ]:
# ------------------------------------------------------------------ run Stage 1 QC
CAL_WL = parse_cal(CAL_FILE)
if CAL_WL.size != NSPEC:
    print(f"cal has {CAL_WL.size} wl (expected {NSPEC}); using channel index"); CAL_WL = np.arange(NSPEC, float)
files = sorted(set(list(SUNA_L0.glob("*.CSV")) + list(SUNA_L0.glob("*.csv"))))
FRAMES, SPECTRA, inv = {}, {}, []
for p in files:
    fhr, df, sp = parse_suna_file(p)
    if df.empty: print("  ", p.name, "no frames"); continue
    df = qc_frames(df, sp, CAL_WL)
    FRAMES[p.name] = df; SPECTRA[p.name] = sp
    # write L1 table (drop raw spectra; those stay in L0)
    keep = (["time_utc","nitrate_uM","abs254","abs350","spec_avg","peak_counts","dark",
             "longwave_A","shape_dev","fit_rmse","T_lamp","humidity","L1_clean"] + ALL_FLAGS)
    df[keep].to_csv(L1_DIR / f"{p.stem}__L1.csv", index=False, encoding="utf-8-sig")
    row = dict(file=p.name, n=len(df), clean=int(df["L1_clean"].sum()))
    row["clean_pct"] = round(100*row["clean"]/row["n"], 1)
    for fl in ALL_FLAGS: row[fl.replace("f_","")] = int(df[fl].sum())
    inv.append(row)
qc = pd.DataFrame(inv)
if not qc.empty:
    qc.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_L1_qc_summary.csv", index=False, encoding="utf-8-sig")
    with pd.option_context("display.width", 200, "display.max_columns", 50):
        print(qc.to_string(index=False))
    print(f"\n{len(qc)} files | {int(qc['n'].sum())} frames | "
          f"{int(qc['clean'].sum())} L1-clean ({100*qc['clean'].sum()/qc['n'].sum():.1f}%)")
    print("L1 tables ->", L1_DIR)

In [ ]:
# ------------------------------------------------------------------ per-file QC figure (static PNG)
def qc_png(fname):
    df = FRAMES[fname]; t = (df["time_utc"] - df["time_utc"].iloc[0]).dt.total_seconds().to_numpy()
    good = df["L1_clean"].to_numpy()
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(figsize=(12, 7.0)); FigureCanvasAgg(fig); ax = fig.subplots(3, 1, sharex=True)
    ax[0].plot(t[good], df["nitrate_uM"].to_numpy()[good], ".", ms=1.4, color="#2E6FB0", label="L1 clean")
    ax[0].plot(t[~good], df["nitrate_uM"].to_numpy()[~good], ".", ms=1.6, color="#d62728", label="flagged")
    ax[0].axhline(0, color="#bbb", lw=0.5); ax[0].set_ylabel("NO3 (uM)")
    ax[0].legend(fontsize=7, frameon=False, markerscale=4, ncol=2)
    ax[1].plot(t, df["longwave_A"], lw=0.5, color="#8257B5"); ax[1].axhline(LONGWAVE_A_HI, color="#f0a", lw=0.6, ls="--")
    ax[1].set_ylabel("longwave |A|"); ax[1].set_ylim(0, min(0.5, max(0.12, df["longwave_A"].quantile(0.999))))
    ax[2].plot(t, df["shape_dev"], lw=0.5, color="#C77D11"); ax[2].axhline(SHAPE_DEV_HI, color="#f0a", lw=0.6, ls="--")
    ax[2].set_ylabel("shape dev"); ax[2].set_ylim(0, min(0.1, max(0.02, df["shape_dev"].quantile(0.999))))
    ax[2].set_xlabel("seconds from file start")
    for a in ax:
        for s in ("top","right"): a.spines[s].set_visible(False)
    fig.suptitle(f"{CRUISE_ID}  SUNA L1 QC  -  {fname}  ({int(good.sum())}/{len(df)} clean)", fontsize=11)
    fig.tight_layout(rect=[0,0,1,0.97])
    out = OUT_DIR / f"{CRUISE_ID}__suna_L1__{Path(fname).stem}.png"
    fig.savefig(out, dpi=125)
    try: plt.close(fig)
    except Exception: pass
    return out

if FRAMES:
    for fn in FRAMES:
        png = qc_png(fn)
        if Image is not None:
            try: display(Image(filename=str(png)))
            except Exception: pass
        print("  saved", png.name)

In [ ]:
# ------------------------------------------------------------------ interactive: QC explorer
# Step through frames (or jump to the next flagged one) and see WHY a frame failed: raw spectrum,
# absorbance near the fit window, and the long-wave baseline. Frame title lists every flag that fired.
try:
    import ipywidgets as W
    _HAVE_W = True
except Exception:
    _HAVE_W = False
    print("ipywidgets not installed; pip install ipywidgets ipympl, then Restart Kernel + Run All.")

REF = {}
for _fn, _df in FRAMES.items():
    _l0 = _df[["f_warmup","f_saturated","f_lowsignal","f_highrmse","f_humid"]].any(axis=1).to_numpy()
    REF[_fn] = np.median(SPECTRA[_fn][~_l0], axis=0) if (~_l0).sum() >= 5 else np.median(SPECTRA[_fn], axis=0)

def _qc_png_bytes(fname, i):
    import io
    df = FRAMES[fname]; wl = CAL_WL; i = int(max(0, min(i, len(df) - 1)))
    s = SPECTRA[fname][i]; r = df.iloc[i]; dk = r["dark"]
    num = np.clip(s - dk, 1, None); den = np.clip(REF[fname] - dk, 1, None); A = -np.log10(num / den)
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(figsize=EXPLORER_FIGSIZE); FigureCanvasAgg(fig); ax1, ax2, ax3 = fig.subplots(1, 3)
    ax1.axvspan(FIT_LO, FIT_HI, color="#2E6FB0", alpha=0.10); ax1.plot(wl, s, lw=0.9, color="#333")
    ax1.axhline(dk, color="#c060c0", lw=0.7, ls="--", label=f"dark {dk:.0f}")
    ax1.set_ylim(0, 66000); ax1.set_xlabel("nm"); ax1.set_ylabel("counts"); ax1.set_title("raw", fontsize=9)
    ax1.legend(fontsize=7, frameon=False)
    m = (wl >= 205) & (wl <= 260)
    ax2.axvspan(FIT_LO, FIT_HI, color="#2E6FB0", alpha=0.10, label="fit 217-240")
    ax2.plot(wl[m], A[m], lw=1.2, color="#B00"); ax2.axhline(0, color="#aaa", lw=0.6)
    ax2.set_xlabel("nm"); ax2.set_ylabel("absorbance (rel)"); ax2.set_title("fit window", fontsize=9)
    ax2.legend(fontsize=7, frameon=False)
    mb = (wl >= 290) & (wl <= 370)
    ax3.axvspan(LONGWAVE_LO, LONGWAVE_HI, color="#8257B5", alpha=0.12, label="baseline 310-360")
    ax3.plot(wl[mb], A[mb], lw=1.2, color="#8257B5"); ax3.axhline(0, color="#aaa", lw=0.6)
    ax3.axhline(LONGWAVE_A_HI, color="#f0a", lw=0.6, ls="--"); ax3.axhline(-LONGWAVE_A_HI, color="#f0a", lw=0.6, ls="--")
    ax3.set_xlabel("nm"); ax3.set_ylabel("absorbance (rel)"); ax3.set_title("long-wave baseline", fontsize=9)
    ax3.legend(fontsize=7, frameon=False)
    for ax in (ax1, ax2, ax3):
        for s_ in ("top","right"): ax.spines[s_].set_visible(False)
    flags = [k[2:] for k in ALL_FLAGS if r.get(k)]
    verdict = "CLEAN" if r["L1_clean"] else "FLAGGED"
    ttl = (f"{fname}  frame {i}/{len(df)-1}  @ {r['time_utc'].strftime('%H:%M:%S')}   "
           f"NO3 {r['nitrate_uM']:.2f}   lwA {r['longwave_A']:.3f}   shape {r['shape_dev']:.4f}   [{verdict}]")
    if flags: ttl += "   " + ",".join(flags)
    fig.suptitle(ttl, fontsize=9.5, color=("#b00" if not r["L1_clean"] else "#1a7f37"))
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=118); buf.seek(0)
    try: plt.close(fig)
    except Exception: pass
    return buf.getvalue()

if _HAVE_W and FRAMES:
    _files = list(FRAMES); _n0 = len(FRAMES[_files[0]]) - 1
    wfile  = W.Dropdown(options=_files, value=_files[0], description="file")
    wframe = W.IntSlider(min=0, max=_n0, value=0, description="frame",
                         continuous_update=False, layout=W.Layout(width="55%"))
    wnum   = W.BoundedIntText(min=0, max=_n0, value=0, description="go to", layout=W.Layout(width="150px"))
    bprev  = W.Button(description="< prev", layout=W.Layout(width="78px"))
    bnext  = W.Button(description="next >", layout=W.Layout(width="78px"))
    bflag  = W.Button(description="next flagged >>", button_style="warning", layout=W.Layout(width="140px"))
    W.link((wframe, "value"), (wnum, "value"))
    def _sync(*_):
        n = len(FRAMES[wfile.value]) - 1; wframe.max = n; wnum.max = n
        if wframe.value > n: wframe.value = 0
    wfile.observe(_sync, "value")
    bprev.on_click(lambda _: setattr(wframe, "value", max(0, wframe.value - 1)))
    bnext.on_click(lambda _: setattr(wframe, "value", min(wframe.max, wframe.value + 1)))
    def _next_flag(_):
        bad = (~FRAMES[wfile.value]["L1_clean"]).to_numpy()
        nxt = np.where(bad[wframe.value + 1:])[0]
        if len(nxt): wframe.value = int(wframe.value + 1 + nxt[0])
    bflag.on_click(_next_flag)
    def _render(fname, i): display(Image(data=_qc_png_bytes(fname, i)))
    out = W.interactive_output(_render, {"fname": wfile, "i": wframe})
    display(W.VBox([W.HBox([wfile, bprev, bnext, bflag]), W.HBox([wframe, wnum]), out]))